# Mixed-data SFT (A: 4000 human, B: + 3x AI) + eval — Colab free T4
Runtime → GPU (T4). Put `brickexp_code.zip` in `MyDrive/brickexp_mix/` (or upload to /content). Run all; after a
disconnect just Run all again — finished stages are skipped and training resumes from its last checkpoint.
Order: train B → eval B → eval base → train A → eval A. Checker = connectivity (as C1); Gurobi re-judge locally.

In [ ]:
import os, sys, glob, json, math, time, zipfile, subprocess
OUTD = "/kaggle/working"
os.makedirs(OUTD, exist_ok=True); T0 = time.time()
def stage(n):
    print(f"=== [{(time.time()-T0)/60:6.1f} min] {n}", flush=True)
    open(f"{OUTD}/stage_{time.strftime('%m%d_%H%M%S')}_{n.replace(' ', '_')}.txt", "w").write(time.ctime())
def run_logged(cmd, tag):
    stage(tag)
    p = subprocess.run(f"set -o pipefail; ( {cmd} ) 2>&1 | tee -a {OUTD}/{tag}.log", shell=True, executable="/bin/bash",
                       capture_output=True, text=True)
    print(p.stdout[-4000:])
    if p.returncode != 0:
        open(f"{OUTD}/error.txt", "a").write(f"[{tag}] rc={p.returncode}\n" + p.stdout[-3000:])
        raise RuntimeError(f"{tag} failed rc={p.returncode}")
    return p
stage("pip")
r = subprocess.run("pip install -q vllm peft gurobipy datasets networkx bitsandbytes", shell=True, capture_output=True, text=True)
print(r.returncode, r.stderr[-1500:])
subprocess.run("pip uninstall -y -q torchao", shell=True)
print(subprocess.run("pip list 2>/dev/null | grep -iE '^(vllm|torch|transformers|peft) '", shell=True, capture_output=True, text=True).stdout)

In [ ]:
WORK = "/kaggle/temp/RLtheLEGO"  # code on local disk, outputs in OUTD
zips = glob.glob("/kaggle/input/**/brickexp_code.zip", recursive=True)
if not os.path.exists(WORK + "/brickexp"):
    dirs = [os.path.dirname(os.path.dirname(p)) for p in glob.glob("/kaggle/input/**/brickexp/__init__.py", recursive=True)]
    if zips:
        zipfile.ZipFile(zips[0]).extractall(WORK)
    elif dirs:
        import shutil; os.makedirs(os.path.dirname(WORK), exist_ok=True); shutil.copytree(dirs[0], WORK, dirs_exist_ok=True)
    else:
        raise RuntimeError("brickexp code not found (zip or extracted dataset)")
os.chdir(WORK); sys.path.insert(0, WORK)
man = json.load(open("data/mix/manifest.json")); print(man)
nB = sum(1 for _ in open("data/mix/B.jsonl")); nA = sum(1 for _ in open("data/mix/A.jsonl"))
STEPS = (math.ceil(nB / 16)) * 3
print("A", nA, "B", nB, "optimizer steps per group:", STEPS)

In [ ]:
def done(p): return os.path.exists(f"{OUTD}/{p}")
def train(g):
    if done(f"sft_{g}/train_log.json") and (done(f"sft_{g}/merged/config.json") or done(f"runs/summary_{g}_newcat.json")):
        print(f"train {g}: done, skipping"); return
    cmd = (f"export CUDA_VISIBLE_DEVICES=0 PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True; "
           f"python -m brickexp.train_sft --model Qwen/Qwen2.5-1.5B-Instruct --data_jsonl data/mix/{g.split('_')[0]}.jsonl --out {OUTD}/sft_{g} --seed 0 "
           f"--bs 2 --grad_accum 8 --max_len 3072 --rank 16 --lr 2e-4 --max_steps {STEPS} --save_steps 40")
    if not done(f"sft_{g}/probe_ok"):
        pr = run_logged(cmd + " --probe_steps 3 --no_merge", f"train_{g}_probe")
        import re; m = re.search(r"PEAK_MEM_GB ([0-9.]+)", pr.stdout); peak = float(m.group(1)) if m else None
        assert peak is None or peak < 13.5, f"would OOM on T4: {peak}"
        os.makedirs(f"{OUTD}/sft_{g}", exist_ok=True); open(f"{OUTD}/sft_{g}/probe_ok", "w").write(str(peak))
    run_logged(cmd, f"train_{g}")
    L = json.load(open(f"{OUTD}/sft_{g}/train_log.json"))
    print("loss:", [(h.get("step"), round(h["loss"], 4)) for h in L["log_history"] if "loss" in h][-10:])
def evaluate(tag, model):
    for suite, extra, tb in (("90", "", 1.5), ("newcat", " --eval_jsonl data/mix/newcat_test_eval.jsonl", 0.5)):
        if done(f"runs/summary_{tag}_{suite}.json"):
            print(f"eval {tag} {suite}: done, skipping"); continue
        run_logged(f"python -m brickexp.eval_2x2 run --backend vllm --max_budget 0 --checker connectivity --limit 0 --out_dir {OUTD}/runs --time_budget_h {tb} --model {model} --tag {tag}_{suite}{extra}", f"eval_{tag}_{suite}")

In [ ]:
train("B_s0_x3"); evaluate("B_s0_x3", f"{OUTD}/sft_B_s0_x3/merged")
if done("runs/summary_B_s0_x3_newcat.json"): subprocess.run(f"rm -rf {OUTD}/sft_B_s0_x3/merged {OUTD}/sft_B_s0_x3/ckpt", shell=True)

In [ ]:
S = " ".join(sorted(p.replace(chr(92), "/") for p in glob.glob(f"{OUTD}/runs/summary_*.json")))
print(run_logged(f"python -m brickexp.eval_2x2 table {S} --out {OUTD}/TABLE_mix.md", "table").stdout)
stage("done")